In [0]:
from pyspark.sql.functions import (
    trunc, col, sum, count, avg, round,
    year, month, date_format
)

In [0]:
# Load GOLD tables

fact = spark.table("gold.fact_transactions")
dim_date = spark.table("gold.dim_date")
dim_merchant = spark.table("gold.dim_merchant")
rfm = spark.table("gold.client_rfm_metrics")

In [0]:
revenue_trend = (
    fact
    .join(dim_date, "date_sk", "left")
    .withColumn("month_date", trunc(col("date"), "month"))
    .groupBy("month_date")
    .agg(
        sum("amount").alias("total_revenue"),
        count("*").alias("transaction_count"),
        avg("amount").alias("avg_transaction_value")
    )
    .withColumn("avg_transaction_value", round(col("avg_transaction_value"), 2))
)

In [0]:
dashboard_rfm = rfm.select(
    "client_sk",
    "R_score",
    "F_score",
    "M_score",
    "rf_segment",
    "rf_segment_label",
    "recency_days",
    "frequency",
    "monetary",
    "avg_transaction_value",
    "customer_lifetime_value",
    "is_churned",
    "churn_reason"
)

In [0]:
merchant_perf = (
    fact
    .join(dim_merchant, "merchant_sk", "left")
    .groupBy(
        "merchant_sk",
        "merchant_id",
        "merchant_city",
        "merchant_state",
        "mcc"
    )
    .agg(
        sum("amount").alias("total_revenue"),
        count("*").alias("transaction_count"),
        avg("amount").alias("avg_ticket")
    )
)

In [0]:
merchant_perf = merchant_perf.withColumn(
    "avg_ticket",
    round(col("avg_ticket"), 2)
)

In [0]:
customer_activity = rfm.select(
    "client_sk",
    "recency_days",
    "frequency",
    "monetary",
    "avg_transaction_value",
    "customer_lifetime_value",
    "is_churned",
    "churn_reason",
    "rf_segment_label"
)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS analytics")

DataFrame[]

In [0]:
revenue_trend.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("analytics.dashboard_revenue_trend")

dashboard_rfm.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("analytics.dashboard_rfm_segments")

merchant_perf.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("analytics.dashboard_merchant_performance")

customer_activity.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("analytics.dashboard_customer_activity")